# Text mining społeczności muzycznych Reddita

Notebook przedstawia analizę NLP komentarzy pochodzących z wybranych subredditów muzycznych. Celem analizy jest porównanie języka, sentymentu oraz głównych tematów dyskusji w społecznościach skupionych wokół różnych gatunków muzycznych.

Analiza obejmuje trzy główne obszary: preprocessing tekstu, identyfikację charakterystycznych słów i fraz, analizę sentymentu oraz modelowanie tematów z wykorzystaniem LDA.

## 1. Przygotowanie środowiska

W tej części importowane są biblioteki, definiowane ścieżki zapisu wyników oraz ustawiany jest jednolity styl wizualizacji. Notebook zapisuje tabele do folderu `../outputs/reports`, a wykresy do folderu `../outputs/figures`.

In [ ]:
from pathlib import Path
from collections import Counter
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

warnings.filterwarnings("ignore")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

try:
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    USE_SPACY = True
except Exception:
    nlp = None
    USE_SPACY = False

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

DATA_PATH = Path("../data/processed/all_subreddits_sample.csv")
FIGURES_DIR = Path("../outputs/figures")
REPORTS_DIR = Path("../outputs/reports")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model językowy spaCy dostępny: {USE_SPACY}")

## 2. Wczytanie danych

Dane zawierają komentarze z wybranych subredditów muzycznych. W dalszej części notebooka zakładane jest, że podstawowymi kolumnami są co najmniej: `subreddit`, `body` oraz `author`.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

required_columns = {"subreddit", "body", "author"}
missing_columns = required_columns - set(df_raw.columns)
if missing_columns:
    raise ValueError(f"Brakuje wymaganych kolumn: {missing_columns}")

basic_info = pd.DataFrame({
    "metryka": ["liczba wierszy", "liczba kolumn", "liczba subredditów"],
    "wartość": [len(df_raw), df_raw.shape[1], df_raw["subreddit"].nunique()]
})

display(basic_info)
display(df_raw.head())

## 3. Opis procesu preprocessingowego

Preprocessing został uporządkowany tak, aby wszystkie analizy opierały się na tym samym, oczyszczonym zbiorze danych. Kolejność działań jest następująca:

1. usunięcie botów oraz kont automatycznych,
2. usunięcie komentarzy technicznych, usuniętych i niedostępnych (`[deleted]`, `[removed]`),
3. zamiana tekstu na małe litery,
4. usunięcie adresów URL,
5. usunięcie znaków specjalnych i nadmiarowych spacji,
6. tokenizacja,
7. usunięcie stopwords oraz dodatkowych słów o niskiej wartości analitycznej,
8. lematyzacja z użyciem spaCy, a w przypadku braku modelu językowego — stemming z użyciem `PorterStemmer`.

Taki układ ogranicza wpływ szumu technicznego na późniejsze wyniki TF-IDF, sentymentu oraz LDA.

In [ ]:
BOTS = {
    "MusicMirrorMan", "HHHRobot", "NoGoogleAMPBot", "ReconEG", "AutoModerator",
    "RemindMeBot", "SaveVideoBot", "DownloadVideoBot", "LinkifyBot", "auddbot",
    "songbot", "lyricsbot", "MusicIdentifierBot", "SpotifyPreviewBot", "RepostSleuthBot",
    "MAGIC_EYE_BOT", "DuplicateDestroyer", "GrammarBotElite", "ShakespeareBot",
    "WikipediaSummaryBot", "HelpfulBot", "AssistantBot", "FreshReleaseBot", "AlbumDropBot",
    "TrackReleaseBot", "MusicNewsBot", "HipHopBot", "IndieBot", "PopMusicBot",
    "TwitterStatusBot", "YouTubeBot", "RedditVideoBot", "TotesMessenger", "AntiSpamBot",
    "BotDefense"
}

MODERATION_PATTERNS = [
    r"hello, thanks for posting",
    r"your submission has been removed",
    r"this post has been removed",
    r"i am a bot",
    r"beep boop",
    r"automoderator",
    r"please contact the moderators",
]

base_stopwords = set(stopwords.words("english"))
extra_stopwords = {
    "like", "just", "get", "got", "one", "know", "think", "really", "would",
    "also", "even", "much", "still", "way", "good", "make", "people", "time",
    "music", "song", "songs", "album", "albums", "artist", "artists", "track", "tracks",
    "band", "bands", "https", "http", "www", "com", "reddit", "amp", "subreddit"
}
STOP_WORDS = base_stopwords | extra_stopwords
stemmer = PorterStemmer()


def remove_urls(text: str) -> str:
    return re.sub(r"http\S+|www\.\S+", " ", text)


def clean_text_basic(text: str) -> str:
    text = str(text).lower()
    text = remove_urls(text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_texts(texts: pd.Series) -> list[list[str]]:
    cleaned_texts = texts.fillna("").astype(str).map(clean_text_basic).tolist()

    if USE_SPACY:
        processed = []
        for doc in nlp.pipe(cleaned_texts, batch_size=500):
            tokens = [
                token.lemma_.lower()
                for token in doc
                if token.is_alpha
                and token.lemma_.lower() not in STOP_WORDS
                and len(token.lemma_) > 2
            ]
            processed.append(tokens)
        return processed

    processed = []
    for text in cleaned_texts:
        tokens = [
            stemmer.stem(token)
            for token in word_tokenize(text)
            if token.isalpha()
            and token not in STOP_WORDS
            and len(token) > 2
        ]
        processed.append(tokens)
    return processed


def filter_dataset(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df.copy()
    filtered["body"] = filtered["body"].fillna("").astype(str)
    filtered["author"] = filtered["author"].fillna("unknown").astype(str)

    removed_mask = filtered["body"].str.strip().str.lower().isin(["[deleted]", "[removed]", "deleted", "removed", ""])
    bot_mask = filtered["author"].isin(BOTS) | filtered["author"].str.lower().str.contains("bot", na=False)
    moderation_mask = filtered["body"].str.contains("|".join(MODERATION_PATTERNS), case=False, na=False, regex=True)

    filtered = filtered.loc[~removed_mask & ~bot_mask & ~moderation_mask].copy()
    filtered["clean_text"] = filtered["body"].map(clean_text_basic)
    filtered = filtered[filtered["clean_text"].str.len() > 0].copy()
    filtered["tokens"] = preprocess_texts(filtered["body"])
    filtered["processed_text"] = filtered["tokens"].map(lambda tokens: " ".join(tokens))
    filtered["token_count"] = filtered["tokens"].map(len)
    filtered = filtered[filtered["token_count"] > 0].copy()
    return filtered.reset_index(drop=True)


df = filter_dataset(df_raw)

preprocessing_summary = pd.DataFrame({
    "etap": ["dane początkowe", "dane po czyszczeniu", "usunięte rekordy"],
    "liczba_komentarzy": [len(df_raw), len(df), len(df_raw) - len(df)]
})

preprocessing_summary.to_csv(REPORTS_DIR / "text_mining_preprocessing_summary.csv", index=False)
display(preprocessing_summary)
display(df[["subreddit", "author", "body", "tokens"]].head())

### Komentarz metodologiczny

Czyszczenie danych zostało wykonane przed właściwą analizą tekstu, dzięki czemu wyniki nie są zniekształcane przez automatyczne komentarze botów, komunikaty moderacyjne oraz usunięte treści. Jest to szczególnie ważne w analizie Reddita, ponieważ boty i automatyczne odpowiedzi mogą sztucznie zwiększać częstość określonych słów lub fraz.

## 4. Najczęstsze słowa i bigramy

Pierwszy etap analizy tekstowej obejmuje identyfikację najczęściej występujących słów i bigramów w poszczególnych społecznościach. Ta część pozwala uchwycić podstawowe różnice słownikowe między subredditami.

In [ ]:
def get_top_items_by_subreddit(df: pd.DataFrame, column: str, top_n: int = 15) -> pd.DataFrame:
    rows = []
    for subreddit, group in df.groupby("subreddit"):
        all_items = []
        for items in group[column]:
            all_items.extend(items)
        for item, count in Counter(all_items).most_common(top_n):
            rows.append({"subreddit": subreddit, "item": item, "count": count})
    return pd.DataFrame(rows)


top_words = get_top_items_by_subreddit(df, "tokens", top_n=15)
top_words.to_csv(REPORTS_DIR / "top_words_by_subreddit.csv", index=False)
display(top_words.head(20))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (subreddit, group) in zip(axes, top_words.groupby("subreddit")):
    plot_data = group.sort_values("count", ascending=True)
    ax.barh(plot_data["item"], plot_data["count"])
    ax.set_title(f"r/{subreddit} — najczęstsze słowa")
    ax.set_xlabel("Liczba wystąpień")
    ax.set_ylabel("")

for ax in axes[len(top_words["subreddit"].unique()):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_mining_top_words.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["bigrams"] = df["tokens"].map(lambda tokens: [" ".join(bg) for bg in ngrams(tokens, 2)])
top_bigrams = get_top_items_by_subreddit(df, "bigrams", top_n=15)
top_bigrams.to_csv(REPORTS_DIR / "top_bigrams_by_subreddit.csv", index=False)
display(top_bigrams.head(20))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (subreddit, group) in zip(axes, top_bigrams.groupby("subreddit")):
    plot_data = group.sort_values("count", ascending=True)
    ax.barh(plot_data["item"], plot_data["count"])
    ax.set_title(f"r/{subreddit} — najczęstsze bigramy")
    ax.set_xlabel("Liczba wystąpień")
    ax.set_ylabel("")

for ax in axes[len(top_bigrams["subreddit"].unique()):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_mining_top_bigrams.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretacja najczęstszych słów i bigramów

Najczęstsze słowa pokazują ogólny zakres tematów obecnych w komentarzach, natomiast bigramy pozwalają lepiej uchwycić kontekst wypowiedzi. W analizie społeczności muzycznych szczególnie istotne są powtarzające się nazwy wykonawców, gatunków, wydarzeń, wydań płytowych oraz określenia wartościujące. Różnice między subredditami mogą wskazywać na odmienne style dyskusji: część społeczności może częściej koncentrować się na konkretnych artystach, inne na rekomendacjach, premierach lub ocenach estetycznych.

## 5. Analiza TF-IDF

Sama częstość występowania słów nie zawsze pokazuje, które wyrazy są charakterystyczne dla danej społeczności. Dlatego zastosowano TF-IDF, który podwyższa wagę słów częstych w danym subreddicie, ale relatywnie rzadszych w pozostałych częściach korpusu.

In [ ]:
subreddit_documents = (
    df.groupby("subreddit")["processed_text"]
    .apply(lambda texts: " ".join(texts))
    .reset_index()
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(subreddit_documents["processed_text"])
feature_names = np.array(vectorizer.get_feature_names_out())

rows = []
for idx, subreddit in enumerate(subreddit_documents["subreddit"]):
    scores = tfidf_matrix[idx].toarray().ravel()
    top_indices = scores.argsort()[::-1][:20]
    for rank, feature_idx in enumerate(top_indices, start=1):
        rows.append({
            "subreddit": subreddit,
            "rank": rank,
            "term": feature_names[feature_idx],
            "tfidf_score": round(float(scores[feature_idx]), 4)
        })

tfidf_results = pd.DataFrame(rows)
tfidf_results.to_csv(REPORTS_DIR / "tfidf_characteristic_terms_by_subreddit.csv", index=False)
display(tfidf_results.head(40))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (subreddit, group) in zip(axes, tfidf_results.groupby("subreddit")):
    plot_data = group.head(12).sort_values("tfidf_score", ascending=True)
    ax.barh(plot_data["term"], plot_data["tfidf_score"])
    ax.set_title(f"r/{subreddit} — charakterystyczne terminy TF-IDF")
    ax.set_xlabel("Waga TF-IDF")
    ax.set_ylabel("")

for ax in axes[len(tfidf_results["subreddit"].unique()):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_mining_tfidf_by_subreddit.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretacja TF-IDF

TF-IDF pozwala przejść od ogólnej popularności słów do identyfikacji terminów wyróżniających konkretne społeczności. Jeżeli dany subreddit ma wysokie wartości TF-IDF dla nazw gatunków, artystów albo określonych typów wypowiedzi, można traktować to jako sygnał jego specyficznego profilu tematycznego. W porównaniu z prostą listą najczęstszych słów analiza TF-IDF lepiej pokazuje różnice między fandomami.

## 6. Word clouds

Chmury słów służą jako wizualne podsumowanie słownictwa dominującego w poszczególnych subredditach. Nie zastępują one analizy ilościowej, ale ułatwiają szybkie porównanie profili językowych społeczności.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, (subreddit, group) in zip(axes, df.groupby("subreddit")):
    text = " ".join(group["processed_text"])
    wordcloud = WordCloud(width=900, height=500, background_color="white", max_words=100).generate(text)
    ax.imshow(wordcloud, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"r/{subreddit}")

for ax in axes[df["subreddit"].nunique():]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_mining_wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretacja chmur słów

Chmury słów pokazują ogólny obraz języka używanego w społecznościach, ale należy interpretować je ostrożnie. Wielkość słowa wynika z częstości jego występowania, a nie z jego znaczenia w dyskusji. Dlatego chmury słów najlepiej traktować jako uzupełnienie tabel i wykresów częstości oraz TF-IDF.

## 7. Analiza sentymentu VADER

Do analizy sentymentu wykorzystano VADER, czyli narzędzie przystosowane do krótkich wypowiedzi internetowych. Wynik `compound` przyjmuje wartości od -1 do 1, gdzie wartości ujemne oznaczają wypowiedzi bardziej negatywne, wartości dodatnie bardziej pozytywne, a okolice zera wskazują neutralność.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

df["sentiment"] = df["body"].map(lambda text: analyzer.polarity_scores(str(text))["compound"])


def classify_sentiment(score: float) -> str:
    if score >= 0.05:
        return "positive"
    if score <= -0.05:
        return "negative"
    return "neutral"


df["sentiment_label"] = df["sentiment"].map(classify_sentiment)

sentiment_stats = (
    df.groupby("subreddit")["sentiment"]
    .agg(mean="mean", median="median", std="std", min="min", max="max", count="count")
    .round(3)
    .reset_index()
)

sentiment_distribution = (
    df.groupby(["subreddit", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)

sentiment_distribution_percent = (
    sentiment_distribution
    .div(sentiment_distribution.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
)

sentiment_stats.to_csv(REPORTS_DIR / "sentiment_stats_by_subreddit.csv", index=False)
sentiment_distribution_percent.to_csv(REPORTS_DIR / "sentiment_distribution_percent.csv")

display(sentiment_stats)
display(sentiment_distribution_percent)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for subreddit, group in df.groupby("subreddit"):
    sns.kdeplot(data=group, x="sentiment", label=f"r/{subreddit}", ax=axes[0], fill=False)

axes[0].axvline(0, linestyle="--", linewidth=1)
axes[0].set_title("Rozkład sentymentu komentarzy")
axes[0].set_xlabel("VADER compound score")
axes[0].set_ylabel("Gęstość")
axes[0].legend(title="Subreddit")

means = sentiment_stats.sort_values("mean")
axes[1].barh(means["subreddit"], means["mean"])
axes[1].axvline(0, linestyle="--", linewidth=1)
axes[1].set_title("Średni sentyment według subreddita")
axes[1].set_xlabel("Średni VADER compound score")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_mining_sentiment_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sentiment_distribution_percent[[col for col in ["negative", "neutral", "positive"] if col in sentiment_distribution_percent.columns]].plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)
plt.title("Struktura komentarzy według kategorii sentymentu")
plt.xlabel("Subreddit")
plt.ylabel("Udział komentarzy (%)")
plt.xticks(rotation=0)
plt.legend(title="Sentyment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_mining_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretacja sentymentu

Porównanie średnich wartości sentymentu pozwala ocenić, które społeczności używają bardziej pozytywnego lub bardziej krytycznego języka. Rozkład kategorii pozytywnych, neutralnych i negatywnych pokazuje natomiast, czy różnice wynikają z ogólnego przesunięcia nastroju, czy z większego udziału skrajnych wypowiedzi. W kontekście społeczności muzycznych wyniki mogą zależeć od charakteru dyskusji: recenzje, spory o artystów, rekomendacje i reakcje na premiery mogą generować odmienne profile emocjonalne.

## 8. Modelowanie tematów LDA

Do identyfikacji głównych obszarów tematycznych wykorzystano model LDA. Liczba tematów została ustawiona na 7, ponieważ jest to kompromis między czytelnością interpretacji a możliwością uchwycenia kilku różnych wątków dyskusji. Przy mniejszej liczbie tematów model może nadmiernie łączyć różne obszary, a przy większej liczbie tematów wyniki mogą stać się trudniejsze do ręcznej interpretacji.

Dodatkowo obliczana jest miara coherence, która pomaga ocenić spójność tematów. Wyniki LDA należy jednak traktować jako narzędzie eksploracyjne, a nie jednoznaczny podział treści.

In [ ]:
NUM_TOPICS = 7
NUM_WORDS = 10

lda_results = {}
lda_topic_rows = []
coherence_rows = []

def build_lda_for_group(tokens_list: list[list[str]], num_topics: int = NUM_TOPICS):
    tokens_list = [tokens for tokens in tokens_list if len(tokens) > 0]
    dictionary = corpora.Dictionary(tokens_list)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    corpus = [dictionary.doc2bow(tokens) for tokens in tokens_list]

    if len(dictionary) == 0 or len(corpus) == 0:
        return None, None, None, None

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=42,
        passes=10,
        chunksize=2000,
        alpha="auto",
        eta="auto",
        per_word_topics=False,
    )

    coherence_model = CoherenceModel(
        model=lda_model,
        texts=tokens_list,
        dictionary=dictionary,
        coherence="c_v"
    )
    coherence = coherence_model.get_coherence()
    return lda_model, corpus, dictionary, tokens_list, coherence

for subreddit, group in df.groupby("subreddit"):
    result = build_lda_for_group(group["tokens"].tolist())
    if result[0] is None:
        continue

    lda_model, corpus, dictionary, tokens_list, coherence = result
    lda_results[subreddit] = {
        "model": lda_model,
        "corpus": corpus,
        "dictionary": dictionary,
        "tokens_list": tokens_list,
        "coherence": coherence,
    }

    coherence_rows.append({"subreddit": subreddit, "num_topics": NUM_TOPICS, "coherence_cv": round(coherence, 3)})

    for topic_id in range(NUM_TOPICS):
        top_words_topic = lda_model.show_topic(topic_id, topn=NUM_WORDS)
        lda_topic_rows.append({
            "subreddit": subreddit,
            "topic_id": topic_id + 1,
            "proposed_topic_name": f"Temat {topic_id + 1} — do interpretacji ręcznej",
            "top_words": ", ".join([word for word, _ in top_words_topic]),
            "weights": ", ".join([str(round(weight, 4)) for _, weight in top_words_topic])
        })

lda_topics = pd.DataFrame(lda_topic_rows)
lda_coherence = pd.DataFrame(coherence_rows)

lda_topics.to_csv(REPORTS_DIR / "lda_topics_by_subreddit.csv", index=False)
lda_coherence.to_csv(REPORTS_DIR / "lda_coherence_by_subreddit.csv", index=False)

display(lda_coherence)
display(lda_topics)

In [ ]:
if lda_results:
    fig, axes = plt.subplots(NUM_TOPICS, len(lda_results), figsize=(4 * len(lda_results), 2.2 * NUM_TOPICS))

    if len(lda_results) == 1:
        axes = np.array(axes).reshape(NUM_TOPICS, 1)

    for col, (subreddit, result) in enumerate(lda_results.items()):
        lda_model = result["model"]
        for topic_id in range(NUM_TOPICS):
            ax = axes[topic_id, col]
            top = lda_model.show_topic(topic_id, topn=7)
            words = [word for word, _ in top][::-1]
            weights = [weight for _, weight in top][::-1]
            ax.barh(words, weights)
            ax.set_title(f"r/{subreddit} — temat {topic_id + 1}", fontsize=9)
            ax.set_xlabel("Waga", fontsize=8)
            ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "text_mining_lda_topics.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
dominant_topic_rows = []

for subreddit, result in lda_results.items():
    lda_model = result["model"]
    corpus = result["corpus"]

    topic_counts = Counter()
    for bow in corpus:
        topic_distribution = lda_model.get_document_topics(bow)
        if topic_distribution:
            dominant_topic = max(topic_distribution, key=lambda item: item[1])[0] + 1
            topic_counts[dominant_topic] += 1

    total_docs = sum(topic_counts.values())
    for topic_id in range(1, NUM_TOPICS + 1):
        count = topic_counts.get(topic_id, 0)
        dominant_topic_rows.append({
            "subreddit": subreddit,
            "topic_id": topic_id,
            "comment_count": count,
            "share_percent": round((count / total_docs) * 100, 1) if total_docs else 0
        })

dominant_topics = pd.DataFrame(dominant_topic_rows)
dominant_topics.to_csv(REPORTS_DIR / "lda_dominant_topics_by_subreddit.csv", index=False)
display(dominant_topics)

if not dominant_topics.empty:
    pivot = dominant_topics.pivot(index="subreddit", columns="topic_id", values="share_percent")
    pivot.plot(kind="bar", stacked=True, figsize=(12, 6))
    plt.title("Dominujące tematy LDA według subreddita")
    plt.xlabel("Subreddit")
    plt.ylabel("Udział komentarzy (%)")
    plt.xticks(rotation=0)
    plt.legend(title="Temat", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "text_mining_lda_dominant_topics.png", dpi=150, bbox_inches="tight")
    plt.show()

### Interpretacja tematów LDA

Tabela tematów zawiera najważniejsze słowa przypisane do każdego tematu. Kolumna `proposed_topic_name` została celowo przygotowana do ręcznej interpretacji, ponieważ nazwanie tematów wymaga oceny badacza. Przykładowo temat zawierający słowa związane z premierami, datami i wydawnictwami można opisać jako „nowe wydania”, natomiast temat oparty na nazwach wykonawców może wskazywać na dyskusje fanowskie wokół konkretnych artystów.

Rozkład dominujących tematów według subredditów pozwala sprawdzić, czy różne społeczności koncentrują się na podobnych typach dyskusji, czy też mają odmienne profile tematyczne. W interpretacji należy jednak pamiętać, że LDA identyfikuje współwystępowanie słów, a nie pełne znaczenie wypowiedzi.

## 9. Ograniczenia analizy

Analiza text mining ma kilka istotnych ograniczeń:

- VADER dobrze sprawdza się w krótkich tekstach internetowych, ale może słabiej rozpoznawać ironię, sarkazm i kontekst kulturowy.
- Język Reddita zawiera slang, skróty, memy, nazwy własne i odniesienia środowiskowe, które nie zawsze są poprawnie interpretowane przez standardowe narzędzia NLP.
- LDA traktuje dokumenty jako zbiory słów i nie rozumie kolejności zdań ani głębszego kontekstu wypowiedzi.
- Wyniki TF-IDF zależą od zakresu korpusu oraz zastosowanej listy stopwords.
- Ręczne nazwanie tematów LDA ma charakter interpretacyjny i może zależeć od wiedzy badacza o danej społeczności muzycznej.
- Komentarze z Reddita nie muszą reprezentować całej społeczności fanów danego gatunku muzycznego, lecz aktywnych użytkowników konkretnej platformy.

## 10. Wnioski z analizy text mining

Analiza NLP pozwala porównać społeczności muzyczne Reddita pod względem języka, nastroju i dominujących tematów. Najczęstsze słowa i bigramy pokazują podstawowe różnice w słownictwie, natomiast TF-IDF wskazuje terminy szczególnie charakterystyczne dla poszczególnych subredditów. Analiza sentymentu umożliwia ocenę ogólnego tonu wypowiedzi, ale jej wyniki powinny być interpretowane ostrożnie ze względu na ironię i slang. Modelowanie LDA dostarcza dodatkowego wglądu w strukturę tematów, lecz wymaga ręcznego nazwania i krytycznej interpretacji.

Wyniki tej części projektu stanowią podstawę do dalszego porównania społeczności muzycznych oraz mogą zostać zestawione z analizą sieciową, aby sprawdzić, czy różnice językowe i tematyczne wiążą się ze strukturą interakcji między użytkownikami.